In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# Generate synthetic dataset (random sequences and labels)
def generate_data(num_samples=1000, seq_length=10, input_size=5):
    X = torch.rand(num_samples, seq_length, input_size)  # Random sequences
    y = torch.randint(0, 2, (num_samples,))  # Binary classification
    return X, y

# Define the GRU Model
class GRUClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(GRUClassifier, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        out, _ = self.gru(x, h0)
        out = self.fc(out[:, -1, :])  # Take the last time step's output
        return out

# Hyperparameters
input_size = 5
hidden_size = 16
num_layers = 1
output_size = 2
learning_rate = 0.001
num_epochs = 10

# Generate Data
X, y = generate_data()
train_size = int(0.8 * len(X))
X_train, y_train = X[:train_size], y[:train_size]
X_test, y_test = X[train_size:], y[train_size:]

# Initialize Model, Loss, Optimizer
model = GRUClassifier(input_size, hidden_size, num_layers, output_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Training Loop
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

# Evaluation
model.eval()
with torch.no_grad():
    test_outputs = model(X_test)
    predicted = torch.argmax(test_outputs, dim=1)
    accuracy = (predicted == y_test).float().mean()
    print(f'Accuracy: {accuracy.item():.4f}')
